# SemanticEmbed: Quickstart

**6D structural intelligence for directed graphs.**

This notebook walks you through:
1. Installing the SDK
2. Encoding a microservice graph
3. Reading the 6D output
4. Generating a structural risk report

**No GPU needed. Runs in seconds.**

[GitHub](https://github.com/jmurray10/semanticembed-sdk)


In [ ]:
# Install the SDK
!pip install -q semanticembed

from semanticembed import encode, report
print("SemanticEmbed installed.")


---
## Step 1: Define a Graph

The only input is an edge list -- a list of `(source, target)` pairs. This is Google's Online Boutique, an 11-service e-commerce application.


In [ ]:
# Google Online Boutique -- 11 microservices
edges = [
    ("frontend",              "ad-service"),
    ("frontend",              "cart-service"),
    ("frontend",              "checkout-service"),
    ("frontend",              "currency-service"),
    ("frontend",              "product-catalog-service"),
    ("frontend",              "recommendation-service"),
    ("frontend",              "shipping-service"),
    ("checkout-service",      "cart-service"),
    ("checkout-service",      "currency-service"),
    ("checkout-service",      "email-service"),
    ("checkout-service",      "payment-service"),
    ("checkout-service",      "product-catalog-service"),
    ("checkout-service",      "shipping-service"),
    ("recommendation-service","product-catalog-service"),
    ("cart-service",          "redis-cart"),
]

print(f"Edges: {len(edges)}")
for src, tgt in edges:
    print(f"  {src} -> {tgt}")


---
## Step 2: Compute the 6D Encoding

One function call. Sub-millisecond.


In [ ]:
result = encode(edges)

print(f"Encoding time: {result.encoding_time_ms:.2f}ms")
print(f"Nodes: {result.graph_info['nodes']}")
print(f"Edges: {result.graph_info['edges']}")
print(f"Max depth: {result.graph_info['max_depth']}")
print()
print(result.table)


---
## Step 3: Understand the Output

Each node now has six numbers -- its coordinates in 6-dimensional structural space:

| Dimension | What It Measures |
|-----------|-----------------|
| **Depth** | Position in the execution pipeline (0.0 = entry, 1.0 = deepest) |
| **Independence** | Lateral redundancy at this pipeline stage (0.0 = only node at this depth) |
| **Hierarchy** | Module / group membership |
| **Throughput** | Fraction of total traffic flowing through this node |
| **Criticality** | Fraction of end-to-end paths that depend on this node |
| **Fanout** | Broadcaster (1.0) vs aggregator (0.0) |

These six dimensions are mathematically independent. Knowing any five tells you nothing about the sixth.


In [ ]:
# Inspect specific nodes
dim_names = ["depth", "independence", "hierarchy", "throughput", "criticality", "fanout"]

for node in ["frontend", "checkout-service", "redis-cart"]:
    v = result.vectors[node]
    print(f"\n{node}:")
    for name, val in zip(dim_names, v):
        bar = "#" * int(val * 30)
        print(f"  {name:<15} {val:.3f}  {bar}")


---
## Step 4: Structural Risk Report

The encoding identifies architectural risks purely from topology. No runtime data, no historical incidents.


In [ ]:
risk = report(result)
print(risk)


---
## Step 5: Try a Different Graph

Same SDK, different architecture. Weaveworks Sock Shop -- 14 services on Kubernetes.


In [ ]:
sock_shop = [
    ("edge-router",   "frontend"),
    ("frontend",      "catalogue"),
    ("frontend",      "orders"),
    ("frontend",      "user"),
    ("frontend",      "carts"),
    ("orders",        "shipping"),
    ("orders",        "payment"),
    ("orders",        "carts"),
    ("orders",        "user"),
    ("catalogue",     "catalogue-db"),
    ("carts",         "carts-db"),
    ("orders",        "orders-db"),
    ("user",          "user-db"),
    ("shipping",      "rabbitmq"),
    ("queue-master",  "rabbitmq"),
]

result_shop = encode(sock_shop)
print(result_shop.table)
print()
print(report(result_shop))


---
## Step 6: Your Own Graph

Paste your edge list below. Any directed graph -- microservices, data pipelines, CI/CD workflows, agent orchestrations.


In [ ]:
# Replace with your own edges
my_edges = [
    ("service-a", "service-b"),
    ("service-a", "service-c"),
    ("service-b", "service-d"),
    ("service-c", "service-d"),
    ("service-d", "service-e"),
]

my_result = encode(my_edges)
print(my_result.table)
print()
print(report(my_result))


---
## What's Next?

- **[02 - Dimensions Deep Dive](https://colab.research.google.com/github/jmurray10/semanticembed-sdk/blob/main/notebooks/02_dimensions.ipynb)** -- what each dimension means with worked examples
- **[03 - Drift Detection](https://colab.research.google.com/github/jmurray10/semanticembed-sdk/blob/main/notebooks/03_drift_detection.ipynb)** -- compare graph versions, detect structural changes
- **[04 - Bring Your Own Graph](https://colab.research.google.com/github/jmurray10/semanticembed-sdk/blob/main/notebooks/04_bring_your_own.ipynb)** -- load from JSON, OTel traces, or Kubernetes

**Free tier:** 50 nodes. **Paid tier:** unlimited nodes + continuous monitoring.

[GitHub](https://github.com/jmurray10/semanticembed-sdk) | [Contact](mailto:jeffmurr@seas.upenn.edu)

*Patent pending. Application #63/994,075.*
